In [98]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy

In [99]:
data  = pd.read_csv("/content/sample_data/california_housing_test.csv")
print(data.head())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.05     37.37                27.0       3885.0           661.0   
1    -118.30     34.26                43.0       1510.0           310.0   
2    -117.81     33.78                27.0       3589.0           507.0   
3    -118.36     33.82                28.0         67.0            15.0   
4    -119.67     36.33                19.0       1241.0           244.0   

   population  households  median_income  median_house_value  
0      1537.0       606.0         6.6085            344700.0  
1       809.0       277.0         3.5990            176500.0  
2      1484.0       495.0         5.7934            270500.0  
3        49.0        11.0         6.1359            330000.0  
4       850.0       237.0         2.9375             81700.0  


In [100]:
# Drop rows with missing values (if any)
data = data.dropna()

# Features (X) and target (y)
X = data.drop("median_house_value", axis=1)
y = data["median_house_value"]

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [101]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize scalers
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# Scale features
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

# Reshape target before scaling
y_train = y_train.values.reshape(-1, 1)
y_test = y_test.values.reshape(-1, 1)

# Scale target
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# Convert to PyTorch tensors and move to device
X_train = torch.from_numpy(X_train).float().to(device)
X_test = torch.from_numpy(X_test).float().to(device)
y_train = torch.from_numpy(y_train).float().to(device)
y_test = torch.from_numpy(y_test).float().to(device)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([2400, 8]) torch.Size([2400, 1])
torch.Size([600, 8]) torch.Size([600, 1])


In [102]:
class HousingModel(nn.Module):
    def __init__(self, input_size=8, hidden1=64, hidden2=32, output_size=1):
        super(HousingModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [103]:
# Initialize model
model = HousingModel(input_size=X_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [104]:
epochs = 300

for epoch in range(epochs):
    model.train()

    # Forward pass
    outputs = model(X_train)  # X_train already on device
    loss = criterion(outputs, y_train)  # y_train already on device

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [20/300], Loss: 0.7432
Epoch [40/300], Loss: 0.4795
Epoch [60/300], Loss: 0.4110
Epoch [80/300], Loss: 0.3732
Epoch [100/300], Loss: 0.3474
Epoch [120/300], Loss: 0.3277
Epoch [140/300], Loss: 0.3120
Epoch [160/300], Loss: 0.2996
Epoch [180/300], Loss: 0.2895
Epoch [200/300], Loss: 0.2808
Epoch [220/300], Loss: 0.2734
Epoch [240/300], Loss: 0.2671
Epoch [260/300], Loss: 0.2617
Epoch [280/300], Loss: 0.2568
Epoch [300/300], Loss: 0.2521


In [105]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test)
    test_loss = criterion(y_pred, y_test)
    print(f"Test MSE Loss: {test_loss.item():.4f}")

Test MSE Loss: 0.2644


In [107]:
y_pred_rescaled = scaler_y.inverse_transform(y_pred.cpu().numpy())
y_test_rescaled = scaler_y.inverse_transform(y_test.cpu().numpy())

# Show first 5 predictions vs actual
for i in range(5):
    print(f"Predicted: {y_pred_rescaled[i][0]:.2f}, Actual: {y_test_rescaled[i][0]:.2f}")


Predicted: 98701.34, Actual: 119400.00
Predicted: 136683.58, Actual: 133600.00
Predicted: 208700.47, Actual: 173600.00
Predicted: 201300.03, Actual: 218600.00
Predicted: 174378.22, Actual: 276700.00


In [111]:
# Pick a new house from test set
new_house = X_test[0].unsqueeze(0)  # keep batch dimension

with torch.no_grad():
    pred_price = model(new_house)
    pred_price_real = scaler_y.inverse_transform(pred_price.cpu().numpy())

print("Predicted price for new house:", pred_price_real[0][0])


Predicted price for new house: 98701.33
